In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
# data path
DATA_PATH = Path(
    r"C:\Users\jchmo\Downloads\dataset_football_matches\big5"
)

In [3]:
# dictionary of the league names
league_names = {
    "E0": "Premier League",
    "D1": "Bundesliga",
    "I1": "Serie A",
    "SP1": "La Liga",
    "F1": "Ligue 1"
}

In [4]:
# load all the data
dataframes = []

for file in sorted(DATA_PATH.glob("*.csv")):

    # eg. E0_2122
    file_name = file.stem

    # split the name to season and league
    league_code, season_code = file_name.split("_")

    # league name
    league = league_names[league_code]

    # 2122 -> 2021/22
    start_year = 2000 + int(season_code[:2])
    end_year = 2000 + int(season_code[2:])

    season = f"{start_year}/{str(end_year)[-2:]}"

    # load CSV
    try:
        df_temp = pd.read_csv(file, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df_temp = pd.read_csv(file, encoding="cp1252")

    # delete empty columns
    df_temp.columns = df_temp.columns.str.strip()

    # add info about league and season
    df_temp["Season"] = season
    df_temp["League"] = league
    df_temp["LeagueCode"] = league_code

    # save the data to dataframe
    dataframes.append(df_temp)

# concatenate all dataframes
df = pd.concat(
    dataframes,
    ignore_index=True,
    sort=False
)

In [ ]:
# fix the Date, so it is not object
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df = df.dropna(subset=["FTR", "FTHG", "FTAG"])           # only played matches
df = df.sort_values(["Date", "Time"]).reset_index(drop=True)
df["match_id"] = df.index

In [6]:
# create rolling features - long table
home = pd.DataFrame({
    "match_id": df["match_id"],
    "Date":     df["Date"],
    "team":     df["HomeTeam"],
    "is_home":  1,
    "gf":       df["FTHG"],   # goals for
    "ga":       df["FTAG"],   # goals against
    "sot_f":    df["HST"],    # shots on target for
    "sot_a":    df["AST"],    # shots on target against
    "points":   df["FTR"].map({"H": 3, "D": 1, "A": 0}),
})

away = pd.DataFrame({
    "match_id": df["match_id"],
    "Date":     df["Date"],
    "team":     df["AwayTeam"],
    "is_home":  0,
    "gf":       df["FTAG"],
    "ga":       df["FTHG"],
    "sot_f":    df["AST"],
    "sot_a":    df["HST"],
    "points":   df["FTR"].map({"H": 0, "D": 1, "A": 3}),
})

long = pd.concat([home, away])
long = long.sort_values(["team", "Date", "match_id"]).reset_index(drop=True)

In [7]:
# rolling features
stats = ["points", "gf", "ga", "sot_f", "sot_a"]

# Overall form: the team's previous 5 matches, home and away together
g = long.groupby("team")
for c in stats:
    long[f"{c}_last5"] = g[c].transform(
        lambda s: s.shift(1).rolling(5, min_periods=5).mean()
    )

# Venue form: previous 5 HOME games for home teams, previous 5 AWAY games for away teams
g_venue = long.groupby(["team", "is_home"])
for c in stats:
    long[f"{c}_venue5"] = g_venue[c].transform(
        lambda s: s.shift(1).rolling(5, min_periods=5).mean()
    )

# shift(1) drops the current match from its own average.
# min_periods=5 gives NaN until a team has 5 previous matches, rather than averaging fewer games.
# The venue version groups by ["team", "is_home"], so a team’s home games and away games are tracked separately.

In [8]:
# merge back
feat_cols = [c for c in long.columns if c.endswith("_last5") or c.endswith("_venue5")]

home_feats = (long[long["is_home"] == 1][["match_id"] + feat_cols]
              .rename(columns={c: f"home_{c}" for c in feat_cols}))
away_feats = (long[long["is_home"] == 0][["match_id"] + feat_cols]
              .rename(columns={c: f"away_{c}" for c in feat_cols}))

df = df.merge(home_feats, on="match_id").merge(away_feats, on="match_id")

# Differences: home advantage in form, which models often use better than raw values
for c in feat_cols:
    df[f"diff_{c}"] = df[f"home_{c}"] - df[f"away_{c}"]

In [9]:
df.shape

(8908, 196)

In [10]:
def add_elo(df, k=20, hfa=70, start=1500):
    """
    Loops through matches in chronological order and returns the
    PRE-match ratings of both teams for every match.
    """
    ratings = {}                       # current rating of each team
    home_elo_list, away_elo_list = [], []

    # actual result from the home team's view: win=1, draw=0.5, loss=0
    result_map = {"H": 1.0, "D": 0.5, "A": 0.0}

    for home, away, ftr in zip(df["HomeTeam"], df["AwayTeam"], df["FTR"]):
        # 1. look up ratings (unseen teams start at 1500)
        r_home = ratings.get(home, start)
        r_away = ratings.get(away, start)

        # 2. SAVE the pre-match ratings: these are the features
        home_elo_list.append(r_home)
        away_elo_list.append(r_away)

        # 3. expected result for the home team (home advantage added)
        expected = 1 / (1 + 10 ** (-(r_home + hfa - r_away) / 400))

        # 4. compare with what actually happened and update
        actual = result_map[ftr]
        change = k * (actual - expected)
        ratings[home] = r_home + change
        ratings[away] = r_away - change

    return home_elo_list, away_elo_list, ratings

In [11]:
df = df.sort_values(["Date", "Time"]).reset_index(drop=True)   # safety: strict chronological order

home_elo, away_elo, final_ratings = add_elo(df, k=20, hfa=70)

df["home_elo"] = home_elo
df["away_elo"] = away_elo
df["elo_diff"] = df["home_elo"] - df["away_elo"]

In [12]:
# 1. Every team's very first match should have rating exactly 1500
first_home = df.groupby("HomeTeam").head(1)[["HomeTeam", "Date", "home_elo"]]
print(first_home["home_elo"].value_counts().head())

# 2. Final ratings: do the top and bottom teams look sensible?
final = pd.Series(final_ratings).sort_values(ascending=False)
print(final.head(10))
print(final.tail(10))

# 3. Does the rating gap relate to results? (it should, clearly)
df["elo_bin"] = pd.qcut(df["elo_diff"], 5)
print(df.groupby("elo_bin")["FTR"].value_counts(normalize=True).unstack().round(3))

home_elo
1500.000000    63
1491.987936    17
1501.987936    15
1511.987936    13
1501.495453     1
Name: count, dtype: int64
Bayern Munich    1801.204859
Barcelona        1775.459867
Inter            1760.474893
Paris SG         1751.279909
Arsenal          1741.591576
Real Madrid      1741.509224
Man City         1718.272531
Dortmund         1678.431504
Napoli           1678.240110
Roma             1659.633569
dtype: float64
Watford             1357.217719
Sheffield United    1355.718871
Monza               1353.979848
Granada             1353.152606
Burnley             1352.933245
Salernitana         1346.858637
Metz                1342.478488
Sampdoria           1339.953625
Valladolid          1314.025671
Southampton         1288.924544
dtype: float64
FTR                                 A      D      H
elo_bin                                            
(-420.43899999999996, -93.143]  0.547  0.250  0.203
(-93.143, -24.572]              0.391  0.271  0.339
(-24.572, 25.014]          

C:\Users\jchmo\AppData\Local\Temp\ipykernel_29344\2690768446.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("elo_bin")["FTR"].value_counts(normalize=True).unstack().round(3))


In [13]:
# 3. Does the rating gap relate to results? (it should, clearly)
df["elo_bin"] = pd.qcut(df["elo_diff"], 5)
print(df.groupby("elo_bin")["FTR"].value_counts(normalize=True).unstack().round(3))

FTR                                 A      D      H
elo_bin                                            
(-420.43899999999996, -93.143]  0.547  0.250  0.203
(-93.143, -24.572]              0.391  0.271  0.339
(-24.572, 25.014]               0.279  0.283  0.437
(25.014, 94.573]                0.221  0.270  0.509
(94.573, 459.402]               0.116  0.195  0.689


C:\Users\jchmo\AppData\Local\Temp\ipykernel_29344\2173912880.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("elo_bin")["FTR"].value_counts(normalize=True).unstack().round(3))
